# 加减乘除初步体验

## 用户需求
本文档记录了用户开发算子的需求，请根据描述进行需求检测及分析。

---

## 需求-1
请开发一个 PyPTO 动态算子：
- 算子名称：add_abs
- 公式：$ y = a + |b| $
- 规格：

| 类型  | shape  | dtype  |
| ------------ | ------------ | ------------ |
| 输入 a| \[n, d\]  | float32  |
| 输入 b| \[n, d\]  | float32  |
| 输出 y| \[n, d\]  | float32  |

- 精度标准：atol=0.000025, rtol=0.005
- 动态轴：n轴

---

# add_abs 算子

## 概述

`add_abs` 是一个 PyPTO 逐元素（element-wise）动态算子，计算 $y = a + |b|$，即输入 `a` 与输入 `b` 的绝对值逐元素相加。

| 属性 | 值 |
|------|-----|
| 算子名称 | `add_abs` |
| 内核函数 | `add_abs_kernel` |
| 算子类型 | 逐元素（Element-wise） |
| 动态轴 | n 轴（支持运行时可变 n） |
| 源文件 | `add_abs.py`（算子 + 测试一体化） |

---

## 数学公式

$$ y = a + |b| $$

对于任意位置 $(i, j)$：

$$ y[i][j] = a[i][j] + |\ b[i][j]\ | $$

---

## 张量规格

| 参数 | 形状 | 数据类型 | 说明 |
|------|------|----------|------|
| 输入 `a` | `[n, d]` | `float32` | 第一输入张量 |
| 输入 `b` | `[n, d]` | `float32` | 第二输入张量（取绝对值后与 a 相加） |
| 输出 `y` | `[n, d]` | `float32` | 计算结果张量 |

- **n 轴**: 动态轴，支持运行时指定任意正整数值。
- **d 轴**: 固定维度（编译时由输入 shape 决定）。

---

## 精度标准

| 指标 | 值 |
|------|-----|
| 绝对容差 (atol) | `0.000025` |
| 相对容差 (rtol) | `0.005` |

---

## 实现细节

### 内核签名

```python
@pypto.frontend.jit(runtime_options={"run_mode": global_run_mode})
def add_abs_kernel(
    a: pypto.Tensor([], pypto.DT_FP32),
    b: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32)
):
```

- **JIT 编译**: 使用 `pypto.frontend.jit` 装饰器自动编译。
- **动态形状**: 张量声明使用 `Tensor([], ...)` 表示动态 shape 支持。
- **Tile 配置**: `set_vec_tile_shapes(2, 8)` 设置向量 tile 形状。

### 核心计算

```python
# torch
  # y = a + torch.abs(b)

# pypto: 
  # tiling
  # operator
```

**涉及的 PyPTO API**：
| API | 用途 | 约束 |
|-----|------|------|
| `pypto.set_vec_tile_shapes` | 设置基本块tiling | 尾轴必须32B对齐 |
| `pypto.abs` | 计算张量 `b` 的逐元素绝对值 | |
| `pypto.add` | 将 `a` 与 `abs(b)` 逐元素相加 | |

**API 使用指导**：
  https://gitcode.com/cann/pypto/blob/master/docs/api/config/pypto-set_vec_tile_shapes.md
  https://gitcode.com/cann/pypto/blob/master/docs/api/operation/pypto-abs.md
  https://gitcode.com/cann/pypto/blob/master/docs/api/operation/pypto-add.md

### 运行模式

- **NPU 模式** (`--run_mode npu`): 在 Ascend NPU 上真实执行，输出实际计算结果并进行精度校验。
- **SIM 模式** (`--run_mode sim`): 编译 + 代价模型仿真，不执行实际计算（输出为未初始化内存），常用于编译验证和性能预估算。

---

## 快速开始

### 环境准备

```bash
# 设置 NPU 设备 ID
export TILE_FWK_DEVICE_ID=0

# 安装依赖（如未安装）
pip install pypto torch numpy
```

### 运行全部测试（NPU 模式）

```bash
python add_abs.py
```

### 运行仿真模式

```bash
python add_abs.py --run_mode sim
```

### 列出所有可用测试

```bash
python add_abs.py --list
```

### 运行指定测试

```bash
python add_abs.py add_abs::test_add_abs_basic
python add_abs.py add_abs::test_add_abs_dynamic_n
python add_abs.py add_abs::test_add_abs_edge_cases
```

---

## 测试用例

| 测试 ID | 名称 | 说明 |
|---------|------|------|
| `add_abs::test_add_abs_basic` | 基础功能验证 | 固定小张量 `[2, 2]` 验证 $y = a + \|b\|$ 的正确性 |
| `add_abs::test_add_abs_dynamic_n` | 动态 n 轴验证 | 测试 `n ∈ {3, 7, 15}` 运行时动态 shape，验证算子对不同 n 值的正确性 |
| `add_abs::test_add_abs_edge_cases` | 边界场景验证 | 覆盖全零输入、b 全部为负数、大数值等边界情况 |

### 预期输出示例（NPU 模式）

```
============================================================
PyPTO add_abs Operator Tests
============================================================

============================================================
Test: Basic Usage of add_abs Operator
============================================================
Input a:    tensor([[ 1., -2.], [ 3., -4.]])
Input b:    tensor([[-2.,  3.], [-4.,  5.]])
Output:     tensor([[3., 1.], [7., 1.]])
Expected:   tensor([[3., 1.], [7., 1.]])
✓ Basic usage of add_abs operator completed successfully

============================================================
Test: add_abs Operator - Dynamic n-axis
============================================================
  n=  3, d=4: max_diff=0.00000000  ✓
  n=  7, d=4: max_diff=0.00000000  ✓
  n= 15, d=4: max_diff=0.00000000  ✓
✓ Dynamic n-axis test completed successfully

============================================================
Test: add_abs Operator - Edge Cases
============================================================
  [zeros]   a=zeros, b=zeros     ✓
  [neg_b]   a=ones,  b=-3        ✓
  [large]   a=[100,-200], b=[-50,150]  ✓
✓ Edge cases test completed successfully

============================================================
All add_abs tests passed!
============================================================
```

---

## 文件结构

```
kadc_test/01-beginner/elewise_ops/
├── add_abs.md          # 需求规格文档
├── add_abs.py          # 算子实现 + 测试用例（唯一文件）
├── ReadMe.md           # 本文档
└── __pycache__/        # Python 字节码缓存
```

---

## 编译产物

运行后在项目根 `output/` 目录下生成性能分析数据：
- `bubble_analysis.log` — 气泡分析报告
- `merged_swimlane.json` — 泳道图数据（可在 [ui.perfetto.dev](https://ui.perfetto.dev) 或 vscode插件 PyPTO ToolKits 打开）

---

## 已知限制

1. **SIM 模式**: 不会产生实际计算结果（仅编译验证和代价模型仿真），精度校验自动跳过。
2. **NPU 环境依赖**: 需要正确安装 CANN 及相关库（`libhccl.so`、`libc_sec.so`、`torch_npu`）。
3. **Tile 配置**: 当前使用固定 `set_vec_tile_shapes(2, 8)`，尾轴需要32B对齐。

In [ ]:
#!/usr/bin/env python3
# coding: utf-8
# Copyright (c) 2025 Huawei Technologies Co., Ltd.
"""
add_abs Operator: y = a + |b|

This file implements the add_abs PyPTO operator:
  y = a + absolute_value(b)

Inputs:
  - a: float32 tensor, shape [n, d]
  - b: float32 tensor, shape [n, d]
Output:
  - y: float32 tensor, shape [n, d]

Precision: atol=0.000025, rtol=0.005
Dynamic axis: n

Usage:
    python add_abs.py                        # Run all tests
    python add_abs.py --list                 # List available tests
    python add_abs.py --run_mode sim         # Run in simulation mode
    python add_abs.py add_abs::test_add_abs_basic  # Run a specific test
"""

import argparse
import os
import sys
import pypto
import torch
import numpy as np
from numpy.testing import assert_allclose


def _peek_run_mode_from_argv(default: str = "npu") -> str:
    """Read run_mode early so module-level decorators can use it."""
    for idx, arg in enumerate(sys.argv):
        if arg == "--run_mode" and idx + 1 < len(sys.argv):
            value = sys.argv[idx + 1]
            if value in ("npu", "sim"):
                return value
        if arg.startswith("--run_mode="):
            value = arg.split("=", 1)[1]
            if value in ("npu", "sim"):
                return value
    return default


global_run_mode = pypto.RunMode.NPU
if _peek_run_mode_from_argv("npu") == "sim":
    global_run_mode = pypto.RunMode.SIM


def get_device_id():
    """Get and validate TILE_FWK_DEVICE_ID from environment variable.

    Returns:
        int: The device ID if valid, None otherwise.
    """
    if 'TILE_FWK_DEVICE_ID' not in os.environ:
        print("Please set the environment variable TILE_FWK_DEVICE_ID before running:")
        print("  export TILE_FWK_DEVICE_ID=0")
        return None

    try:
        device_id = int(os.environ['TILE_FWK_DEVICE_ID'])
        return device_id
    except ValueError:
        print(f"ERROR: TILE_FWK_DEVICE_ID must be an integer, got: {os.environ['TILE_FWK_DEVICE_ID']}")
        return None


# ============================================================================
# add_abs Kernel Definition
# ============================================================================

@pypto.frontend.jit(
    runtime_options={
        "run_mode": global_run_mode
    },
    debug_options={
        "runtime_debug_mode": 0,
        "compile_debug_mode": 0
    }
)
def add_abs_kernel(
    a: pypto.Tensor([], pypto.DT_FP32),
    b: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32)):
    """add_abs kernel: computes y = a + |b| element-wise.

    Args:
        a: Input tensor A of shape [n, d], dtype float32.
        b: Input tensor B of shape [n, d], dtype float32.
        out: Output tensor of shape [n, d], dtype float32.
    """
    # TODO: 请在下方添加 PyPTO 实现代码
    # 设置 tiling
    ...

    # y = a + |b|
    ...


# ============================================================================
# Test Cases
# ============================================================================

def test_add_abs_basic(device_id: int = None):
    """Test basic usage of add_abs operator: y = a + |b|."""
    print("=" * 60)
    print("Test: Basic Usage of add_abs Operator")
    print("=" * 60)

    device = f'npu:{device_id}' if global_run_mode == pypto.RunMode.NPU and device_id is not None else 'cpu'

    dtype = torch.float32
    a = torch.tensor([[1.0, -2.0], [3.0, -4.0]], dtype=dtype, device=device)
    b = torch.tensor([[-2.0, 3.0], [-4.0, 5.0]], dtype=dtype, device=device)
    # expected: a + |b| = [[1+2, -2+3], [3+4, -4+5]] = [[3, 1], [7, 1]]
    expected = a + torch.abs(b)

    out = torch.empty(a.shape, dtype=dtype, device=device)
    add_abs_kernel(a, b, out)
    if global_run_mode == pypto.RunMode.NPU:
        assert_allclose(out.cpu().numpy(), expected.cpu().numpy(), rtol=0.005, atol=0.000025)
    print(f"Input a:    {a}")
    print(f"Input b:    {b}")
    print(f"Output:     {out}")
    print(f"Expected:   {expected}")
    print("✓ Basic usage of add_abs operator completed successfully")


def test_add_abs_dynamic_n(device_id: int = None):
    """Test add_abs operator with dynamic n-axis: different n sizes at runtime."""
    print("=" * 60)
    print("Test: add_abs Operator - Dynamic n-axis")
    print("=" * 60)

    device = f'npu:{device_id}' if global_run_mode == pypto.RunMode.NPU and device_id is not None else 'cpu'

    dtype = torch.float32
    d = 4

    # Test multiple values of n to verify dynamic axis support
    for n in [3, 7, 15]:
        a = torch.randn(n, d, dtype=dtype, device=device)
        b = torch.randn(n, d, dtype=dtype, device=device)
        expected = a + torch.abs(b)

        out = torch.empty(a.shape, dtype=dtype, device=device)
        add_abs_kernel(a, b, out)
        if global_run_mode == pypto.RunMode.NPU:
            assert_allclose(out.cpu().numpy(), expected.cpu().numpy(), rtol=0.005, atol=0.000025)

        max_diff = np.abs(out.cpu().numpy() - expected.cpu().numpy()).max()
        print(f"  n={n:3d}, d={d}: max_diff={max_diff:.8f}  ✓")

    print("✓ Dynamic n-axis test completed successfully")


def test_add_abs_edge_cases(device_id: int = None):
    """Test add_abs operator with edge cases: zeros, all-negative b, large values."""
    print("=" * 60)
    print("Test: add_abs Operator - Edge Cases")
    print("=" * 60)

    device = f'npu:{device_id}' if global_run_mode == pypto.RunMode.NPU and device_id is not None else 'cpu'
    dtype = torch.float32

    # 1. All zeros
    a = torch.zeros((2, 3), dtype=dtype, device=device)
    b = torch.zeros((2, 3), dtype=dtype, device=device)
    expected = a + torch.abs(b)
    out = torch.empty(a.shape, dtype=dtype, device=device)
    add_abs_kernel(a, b, out)
    if global_run_mode == pypto.RunMode.NPU:
        assert_allclose(out.cpu().numpy(), expected.cpu().numpy(), rtol=0.005, atol=0.000025)
    print("  [zeros]   a=zeros, b=zeros     ✓")

    # 2. b all negative — abs should flip signs
    a = torch.ones((2, 3), dtype=dtype, device=device)
    b = torch.full((2, 3), -3.0, dtype=dtype, device=device)
    expected = a + torch.abs(b)  # ones + 3 = 4 everywhere
    out = torch.empty(a.shape, dtype=dtype, device=device)
    add_abs_kernel(a, b, out)
    if global_run_mode == pypto.RunMode.NPU:
        assert_allclose(out.cpu().numpy(), expected.cpu().numpy(), rtol=0.005, atol=0.000025)
    print("  [neg_b]   a=ones,  b=-3        ✓")

    # 3. Large absolute values
    a = torch.tensor([[100.0, -200.0]], dtype=dtype, device=device)
    b = torch.tensor([[-50.0, 150.0]], dtype=dtype, device=device)
    expected = a + torch.abs(b)  # [[100+50, -200+150]] = [[150, -50]]
    out = torch.empty(a.shape, dtype=dtype, device=device)
    add_abs_kernel(a, b, out)
    if global_run_mode == pypto.RunMode.NPU:
        assert_allclose(out.cpu().numpy(), expected.cpu().numpy(), rtol=0.005, atol=0.000025)
    print("  [large]   a=[100,-200], b=[-50,150]  ✓")

    print("✓ Edge cases test completed successfully")


# ============================================================================
# Main entry point
# ============================================================================

def main():
    """Run add_abs operator tests.

    Usage:
        python add_abs.py                        # Run all tests
        python add_abs.py --list                 # List all available tests
        python add_abs.py add_abs::test_add_abs_basic  # Run a specific case
    """
    parser = argparse.ArgumentParser(
        description="PyPTO add_abs Operator Tests",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog="""
Examples:
  %(prog)s                              Run all tests
  %(prog)s --list                       List all available tests
  %(prog)s add_abs::test_add_abs_basic    Run a specific test
  %(prog)s --run_mode sim               Run in simulation mode
        """
    )
    parser.add_argument(
        'example_id',
        type=str,
        nargs="?",
        help='Run a specific test case (e.g., add_abs::test_add_abs_basic). If omitted, run all tests.'
    )
    parser.add_argument(
        '--list',
        action='store_true',
        help='List all available tests and exit'
    )
    parser.add_argument(
        "--run_mode", "--run-mode",
        nargs="?", type=str, default="npu", choices=["npu", "sim"],
        help='Run mode: "npu" (default) or "sim".'
    )

    args = parser.parse_args()

    # Registry of all test cases
    examples = {
        'add_abs::test_add_abs_basic': {
            'name': 'Basic usage of add_abs operator',
            'description': 'Verify y = a + |b| with a fixed small tensor.',
            'function': test_add_abs_basic
        },
        'add_abs::test_add_abs_dynamic_n': {
            'name': 'Dynamic n-axis test',
            'description': 'Verify the operator handles different n sizes at runtime.',
            'function': test_add_abs_dynamic_n
        },
        'add_abs::test_add_abs_edge_cases': {
            'name': 'Edge cases (zeros, all-negative, large values)',
            'description': 'Verify correctness on boundary inputs.',
            'function': test_add_abs_edge_cases
        },
    }

    if args.list:
        print("\n" + "=" * 60)
        print("Available Tests for add_abs Operator")
        print("=" * 60 + "\n")
        for case_key, ex_info in sorted(examples.items()):
            print(f"  {case_key}")
            print(f"     Name: {ex_info['name']}")
            print(f"     Description: {ex_info['description']}\n")
        return

    # Select tests to run
    if args.example_id:
        if args.example_id not in examples:
            print(f"ERROR: Invalid case '{args.example_id}'")
            print(f"Valid cases are: {', '.join(sorted(examples.keys()))}")
            print("\nUse --list to see all available tests.")
            sys.exit(1)
        examples_to_run = [(args.example_id, examples[args.example_id])]
    else:
        examples_to_run = list(examples.items())

    print("\n" + "=" * 60)
    print("PyPTO add_abs Operator Tests")
    print("=" * 60 + "\n")

    device_id = None
    if args.run_mode == "npu":
        device_id = get_device_id()
        if device_id is None:
            return
        import torch_npu
        torch.npu.set_device(device_id)

    try:
        for case_key, ex_info in examples_to_run:
            ex_info['function'](device_id)

        if len(examples_to_run) > 1:
            print("=" * 60)
            print("All add_abs tests passed!")
            print("=" * 60)

    except Exception as e:
        print(f"\nError: {e}")
        raise


if __name__ == "__main__":
    main()